In [1]:
# SetupEnvironment
import os
# import shutil
import requests
# import zipfile
import io
import pandas as pd
import json
import datetime
import re
import calendar
from bs4 import BeautifulSoup
import time

In [4]:
file = r"D:\B-Project\2025\6800\Technical\12票證資料\2024_2025\臺北捷運每日各站分時OD資料(O)\臺北捷運每日各站分時OD資料(O).csv"
pd.read_csv(file, nrows = 500)

,旅次日期,旅次時段(Ex: 12表示為12時00分至12時59分),票證分類 (N-IC: 非電子票證; IC: 電子票證),電子票證卡種,持卡身分,起站代碼,起站中文名稱,迄站代碼,迄站中文名稱,旅運量,資料代表日期(yyyy-MM-dd)
0,TripDate,TripHour,TicketClass,IDType,HolderType,OriginStationID,OriginStationName,DestinationStationID,DestinationStationName,Volume,InfoDate
1,2024-07-01,8,IC,EasyCard,C09,BR06,麟光,G09 / O05,古亭,1,2024-07-01
2,2024-07-01,15,IC,EasyCard,C01,O12,大橋頭,G09 / O05,古亭,1,2024-07-01
3,2024-07-01,23,IC,EasyCard,A,G07,公館,O50,三重國小,1,2024-07-01
4,2024-07-01,16,IC,EasyCard,C09,O11 / R13,民權西路,G01,新店,1,2024-07-01
...,...,...,...,...,...,...,...,...,...,...,...
495,2024-07-01,13,IC,EasyCard,C01,Y09,秀朗橋,G05,景美,1,2024-07-01
496,2024-07-01,11,IC,EasyCard,C09,G18,南京三民,BR11 / G16,南京復興,2,2024-07-01
497,2024-07-01,17,IC,EasyCard,C02,BL12 / R10,台北車站,BR16,西湖,1,2024-07-01
498,2024-07-01,17,IC,iPASS,C02,R22A,新北投,R28,淡水,1,2024-07-01


In [7]:
def melt_thsrc(df, df2):
    # 定義一個簡化的轉換函數
    def process_df(df, value_name):
        df = pd.melt(df, id_vars=['YYYYMM'], var_name='車站名稱', value_name=value_name)
        df[['年', '月']] = df['YYYYMM'].str.split('-', expand=True)
        df = df.drop(columns='YYYYMM')
        df['年'] = df['年'].astype('int64')
        df['月'] = df['月'].astype('int64')
        df[value_name] = df[value_name].str.replace(',', '').astype('int64')
        return df

    # 處理進站和出站數據
    df = process_df(df, '進站人次')
    df2 = process_df(df2, '出站人次')

    # 合併進站和出站數據
    df_merge = pd.merge(df, df2, on=['年', '月', '車站名稱'], how='outer')

    # 計算天數與平均日運量
    df_merge['天數'] = df_merge.apply(lambda row: calendar.monthrange(row['年'] + 1911, row['月'])[1], axis=1)
    df_merge['進站旅客人數(平均日運量)'] = df_merge['進站人次'] // df_merge['天數']
    df_merge['出站旅客人數(平均日運量)'] = df_merge['出站人次'] // df_merge['天數']
    df_merge['總進出旅客人數(平均日運量)'] = df_merge['進站旅客人數(平均日運量)'] + df_merge['出站旅客人數(平均日運量)']

    # 重新排序欄位
    # df_merge = df_merge.reindex(columns=['年', '月', '運具別名稱', '車站名稱', '車站代碼', 
    #                                      '總進出旅客人數(平均日運量)', '進站旅客人數(平均日運量)', '出站旅客人數(平均日運量)'])
    df_merge['運具別名稱'] = '高鐵'

    return df_merge

def THSR():
    # ===== Step1：爬蟲下載高鐵資料 =====
    print('Step1：爬蟲下載高鐵資料')
    # 目標網址
    url = "https://www.thsrc.com.tw/corp/9571df11-8524-4935-8a46-0d5a72e6bc7c"

    # 發送 HTTP 請求到網頁
    response = requests.get(url)

    # 檢查請求是否成功
    if response.status_code == 200:
        # 解析網頁內容
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 抓取「進站人次」數據
        in_station_tab = soup.find('div', {'id': 'tab-01'})
        if in_station_tab:
            in_station_table = in_station_tab.find('table')
            if in_station_table:
                # 抓取表頭
                in_station_headers = [th.text.strip() for th in in_station_table.find('thead').find_all('th')]

                # 抓取表格內容
                in_station_rows = []
                for tr in in_station_table.find('tbody').find_all('tr'):
                    cells = [tr.find('th').text.strip()]  # 抓取行標題 (年度 / 月份)
                    cells.extend([td.text.strip() for td in tr.find_all('td')])  # 抓取其他欄位
                    in_station_rows.append(cells)

                # 將數據轉換為 DataFrame
                in_station_df = pd.DataFrame(in_station_rows, columns=in_station_headers)

                # 重命名欄位 '年度 / 月份' 為 'YYYYMM'
                in_station_df = in_station_df.rename(columns={'年度 / 月份': 'YYYYMM'})

                # 保存 DataFrame 到 Excel
                # in_station_df.to_excel('thsrc_in_station_data.xlsx', index=False)
            else:
                print("無法找到進站人次的表格")
        else:
            print("無法找到進站人次的標籤")

        # 抓取「出站人次」數據
        out_station_tab = soup.find('div', {'id': 'tab-02'})
        if out_station_tab:
            out_station_table = out_station_tab.find('table')
            if out_station_table:
                # 抓取表頭
                out_station_headers = [th.text.strip() for th in out_station_table.find('thead').find_all('th')]

                # 抓取表格內容
                out_station_rows = []
                for tr in out_station_table.find('tbody').find_all('tr'):
                    cells = [tr.find('th').text.strip()]  # 抓取行標題 (年度 / 月份)
                    cells.extend([td.text.strip() for td in tr.find_all('td')])  # 抓取其他欄位
                    out_station_rows.append(cells)

                # 將數據轉換為 DataFrame
                out_station_df = pd.DataFrame(out_station_rows, columns=out_station_headers)

                # 重命名欄位 '年度 / 月份' 為 'YYYYMM'
                out_station_df = out_station_df.rename(columns={'年度 / 月份': 'YYYYMM'})

                # 保存 DataFrame 到 Excel
                # out_station_df.to_excel('thsrc_out_station_data.xlsx', index=False)
            else:
                print("無法找到出站人次的表格")
        else:
            print("無法找到出站人次的標籤")

    else:
        print(f"無法取得網頁，狀態碼: {response.status_code}")

    # ===== Step2：整合資料 =====
    print('Step2：整合資料')

    thsrc = melt_thsrc(df = in_station_df, df2 = out_station_df)
    thsrc['年'] = thsrc['年'] - 1911
    thsrc = thsrc[thsrc['車站名稱'] != '總計']


    # ===== Step3：輸出資料 =====
    thsrc = thsrc.reindex(columns = ['年', '月', '車站名稱', '進站人次', '出站人次', '進站旅客人數(平均日運量)', '出站旅客人數(平均日運量)', '總進出旅客人數(平均日運量)'])
    # thsrc = thsrc.sort_values(['年', '月'])

    station_order = [
        '南港','台北','板橋','桃園','新竹','苗栗',
        '台中','彰化','雲林','嘉義','台南','左營'
    ]

    # 將「車站名稱」轉成有順序的類別
    thsrc['車站名稱'] = pd.Categorical(
        thsrc['車站名稱'],
        categories=station_order,
        ordered=True
    )

    # 依 年 → 月 → 車站名稱 排序
    thsrc = thsrc.sort_values(['年', '月', '車站名稱'])

    return thsrc

In [8]:
def main():
    df = THSR()
    df.to_excel('高鐵官網資料爬蟲.xlsx', sheet_name='各站進出人次', index = False)

if __name__ == "__main__":
    main()

Step1：爬蟲下載高鐵資料
Step2：整合資料
